Working with the data

In [1]:
import pandas as pd
import json
import numpy as np
from bs4 import BeautifulSoup
import math
import os

In [2]:
def remove_tags(html):
    soup = BeautifulSoup(html, "html.parser")

    # Find and replace <img> tags with their 'alt' attribute
    for img in soup.find_all('img'):
        alt_text = img.get('alt', '')  # default to empty string if 'alt' is None
        if alt_text:
            # Create a new text node
            new_text = soup.new_string(" " + alt_text + " ")
            img.replace_with(new_text)
        else:
            # Remove the image if no alt text
            img.decompose()

    # Remove all script and style elements
    for tag in soup(['script', 'style']):
        tag.decompose()

    # Extract the text, cleaning up any excessive whitespace
    clean_text = ' '.join(soup.stripped_strings)
    return clean_text


In [3]:

dataset_paths = ['assist2009/skill_builder_data_corrected_collapsed.csv', 
 'assist2012/2012-2013-data-with-predictions-4-final.csv', 
 'assist2017/anonymized_full_release_competition_dataset.csv']
datasets = ['assist2009', 'assist2012', 'assist2017']

pb_df = pd.read_csv('./data_subsets/ProblemBodies_23.csv', low_memory=False)

In [4]:
dataset_paths[0] , datasets[0]

('assist2009/skill_builder_data_corrected_collapsed.csv', 'assist2009')

In [5]:
pb_df.head()

,problem_id,problem_code,assistment_id,problem_set_id,problem_set_type,problem_set_name,curriculum,grade_or_subject,unit,link,...,attempts,correct_count,percent_correct,first_cwa,first_cwa_count,second_cwa,second_cwa_count,third_cwa,third_cwa_count,answer
0,1152985.0,PRA5FZU,780070.0,PSAQKFU,NaN,"8.5 Comparing Linear, Exponential, and Quadrat...",Textbook Curricula,Big Ideas Learning,HS: Purple Algebra I (2014),https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Exponential
1,1164390.0,PRA5S4J,789764.0,PSAUDWH,NaN,Variables and Patterns Investigation 1 ACE,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1167935.0,PRA5V8Q,792776.0,PSAUK4X,NaN,Variables and Patterns Investigation 3 ACE,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,33.0,21.0,63.6,18,3.0,15,2.0,0.3,1.0,12
3,1152694.0,PRA5FRU,779822.0,PSATNY9,NaN,9.7 Independent and Dependent Events,Textbook Curricula,Glencoe McGraw-Hill,Grade 7: Course 2 (2015),https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,1217278.0,PRA6768,832131.0,PSAVWDT,NaN,Covering and Surrounding ACE Investigation 4,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32


In [6]:
print(pb_df.columns)

Index(['problem_id', 'problem_code', 'assistment_id', 'problem_set_id',
       'problem_set_type', 'problem_set_name', 'curriculum',
       'grade_or_subject', 'unit', 'link', 'skill_codes', 'skill_names',
       'problem_type', 'problem_body', 'attempts', 'correct_count',
       'percent_correct', 'first_cwa', 'first_cwa_count', 'second_cwa',
       'second_cwa_count', 'third_cwa', 'third_cwa_count', 'answer'],
      dtype='object')


In [7]:
pb_df['assistment_id'][1997:2025],pb_df['problem_set_id'][1997:2025]

(1997    1167675.0
 1998     805977.0
 1999    1164522.0
 2000    1164587.0
 2001     806008.0
 2002     806032.0
 2003    1164531.0
 2004     806038.0
 2005     806039.0
 2006     805994.0
 2007     806027.0
 2008    1164529.0
 2009    1164577.0
 2010    1164534.0
 2011    1164630.0
 2012    1164632.0
 2013     783041.0
 2014     783637.0
 2015    1167684.0
 2016    1164540.0
 2017     782672.0
 2018     782670.0
 2019     782657.0
 2020     782673.0
 2021    1164788.0
 2022     783677.0
 2023     887800.0
 2024    1135972.0
 Name: assistment_id, dtype: float64,
 1997    PSABAG8U
 1998     PSAVHVW
 1999    PSABACAA
 2000    PSABACR3
 2001     PSAVHVW
 2002     PSAVHVW
 2003    PSABACAA
 2004     PSAVHVW
 2005     PSAVHVW
 2006     PSAVHVW
 2007     PSAVHVW
 2008    PSABACAA
 2009    PSABACSM
 2010    PSABACAA
 2011    PSABACSM
 2012    PSABACSM
 2013     PSATV76
 2014     PSATX8Q
 2015    PSABAG82
 2016    PSABACR3
 2017     PSATVYP
 2018     PSATVYP
 2019     PSATVPA
 2020     PSATVY

In [8]:
pb_df['problem_set_type'][1997:2025],pb_df['problem_set_name'][1997:2025]

(1997    NaN
 1998    NaN
 1999    NaN
 2000    NaN
 2001    NaN
 2002    NaN
 2003    NaN
 2004    NaN
 2005    NaN
 2006    NaN
 2007    NaN
 2008    NaN
 2009    NaN
 2010    NaN
 2011    NaN
 2012    NaN
 2013    NaN
 2014    NaN
 2015    NaN
 2016    NaN
 2017    NaN
 2018    NaN
 2019    NaN
 2020    NaN
 2021    NaN
 2022    NaN
 2023    NaN
 2024    NaN
 Name: problem_set_type, dtype: object,
 1997                    Lesson 9 Preparing for Assessment
 1998                       Prime Time Investigation 3 ACE
 1999    6-2 Solving Systems Using Substitution: Key Co...
 2000                           Chapter 8 Mid-Chapter Quiz
 2001                       Prime Time Investigation 3 ACE
 2002                       Prime Time Investigation 3 ACE
 2003    6-2 Solving Systems Using Substitution: Key Co...
 2004                       Prime Time Investigation 3 ACE
 2005                       Prime Time Investigation 3 ACE
 2006                       Prime Time Investigation 3 ACE
 2007 

In [9]:
dataset = datasets[0]
dataset_path = dataset_paths[0]

In [10]:
assist_df = pd.read_csv('raw_data/' + dataset_path, encoding = "ISO-8859-1", low_memory=False)
problem_id_column_name = 'problem_id'

In [11]:
assist_df.head()

,Unnamed: 0,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,4,35450295,220674,70363,33110,51395,1,1,1,4859,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,5,35450311,220674,70363,33196,51481,1,0,14,19813,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


In [12]:
len(assist_df)

346860

In [13]:
questions = assist_df[problem_id_column_name].unique()
questions_w_text = pb_df[pb_df['problem_id'].isin(questions)]

In [14]:
len(questions)

26688

In [15]:
questions_w_text

,problem_id,problem_code,assistment_id,problem_set_id,problem_set_type,problem_set_name,curriculum,grade_or_subject,unit,link,...,attempts,correct_count,percent_correct,first_cwa,first_cwa_count,second_cwa,second_cwa_count,third_cwa,third_cwa_count,answer
2190,119299.0,PRACDD6,62585.0,PSAJ78,NaN,(7.SP.C.7a) Probability of a Single Event Skil...,Skill Builders,Grade 7,(SP) Statistics and Probability,https://app.assistments.org/find/lv/lesson/328...,...,61.0,20.0,32.8,"<img style=""width: 237px; height: 53px;"" src=""...",10.0,"<img style=""width: 204px; height: 22px;"" src=""...",9.0,"<img style=""width: 237px; height: 22px;"" src=""...",8.0,"<img style=""width: 246px; height: 57px;"" src=""..."
2654,97581.0,PRAB54Z,55574.0,PSAGGT,NaN,(7.EE.B.4a) Solving Equations (with Combining ...,Skill Builders,Grade 7,(EE) Expressions and Equations,https://app.assistments.org/find/lv/lesson/328...,...,56.0,26.0,46.4,B),12.0,D),8.0,A),8.0,C)
2930,79936.0,PRABSBM,44247.0,PSAGZU,NaN,(8.G.B.7) Pythagorean Theorem - Finding a Miss...,Skill Builders,Grade 8,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,62.0,28.0,45.2,the sum of the two other sides.,13.0,half the base times the height.,8.0,the square of one of the sides.,6.0,the sum of the squares of the two other sides.
2933,76339.0,PRABQWX,42893.0,PSANGJ,NaN,(6.G.A.1) Area of a Trapezoid Skill Builder,Skill Builders,Grade 6,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45
2941,89068.0,PRABYHY,50210.0,PSAHSU,NaN,(8.G.A.5) Sum of Interior Angles of Triangles ...,Skill Builders,Grade 8,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,52.0,30.0,57.7,135,3.0,180,2.0,1440,2.0,1080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472553,90055.0,PRABZE7,51085.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,226.0,160.0,70.8,0,18.0,-100,5.0,-10,5.0,-1
472614,90238.0,PRABZJ2,51204.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,305.0,253.0,83.0,-1,17.0,-2,5.0,1,5.0,-3
472619,90230.0,PRABZJS,51196.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,203.0,178.0,87.7,-9,2.0,-5,2.0,5,1.0,-6
472622,90174.0,PRABZGY,51140.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,227.0,179.0,78.9,-3,25.0,27,7.0,1,2.0,3


In [16]:
len(questions_w_text)

7942

In [17]:
print(f'Number of total questions in {dataset} : {len(questions)}')
print(f'Number of questions in {dataset} with text: {len(questions_w_text)}')

Number of total questions in assist2009 : 26688
Number of questions in assist2009 with text: 7942


In [18]:
assist_df_temp = assist_df[assist_df[problem_id_column_name].isin(questions_w_text['problem_id'])]
assist_df_temp.to_csv('../data/' + dataset_path)


In [19]:
with open('../data/'+ dataset + '/keyid2idx.json', 'r') as f:
        keyid2idx_pb_subset = json.load(f)
print(f'Loaded keyid2idx_pb_subset for {dataset}')

Loaded keyid2idx_pb_subset for assist2009


In [20]:
keyid2idx_pb_subset

{'questions': {'93383': 0,
  '93407': 1,
  '93400': 2,
  '93419': 3,
  '93420': 4,
  '93415': 5,
  '93423': 6,
  '57695': 7,
  '57647': 8,
  '91522': 9,
  '92390': 10,
  '92379': 11,
  '91144': 12,
  '92361': 13,
  '91124': 14,
  '91127': 15,
  '54003': 16,
  '53991': 17,
  '54071': 18,
  '54015': 19,
  '53987': 20,
  '54193': 21,
  '54035': 22,
  '49276': 23,
  '89737': 24,
  '90210': 25,
  '54051': 26,
  '53317': 27,
  '84817': 28,
  '53999': 29,
  '49286': 30,
  '49270': 31,
  '49290': 32,
  '49265': 33,
  '49278': 34,
  '84724': 35,
  '84974': 36,
  '85826': 37,
  '61100': 38,
  '60111': 39,
  '109008': 40,
  '108966': 41,
  '108924': 42,
  '109011': 43,
  '108957': 44,
  '87461': 45,
  '89706': 46,
  '90237': 47,
  '86200': 48,
  '86188': 49,
  '86157': 50,
  '86160': 51,
  '54298': 52,
  '66079': 53,
  '53327': 54,
  '78930': 55,
  '78902': 56,
  '79063': 57,
  '79056': 58,
  '78866': 59,
  '79004': 60,
  '79072': 61,
  '84605': 62,
  '80882': 63,
  '80714': 64,
  '80738': 65,
  

In [21]:
problem_ids = list(keyid2idx_pb_subset['questions'].keys())
problem_ids_int = [int(x) for x in problem_ids]
print(f'Number of problems in {dataset}: {len(problem_ids_int)}')

Number of problems in assist2009: 6035


In [22]:
subset_df_html_selected = pb_df[pb_df['problem_id'].isin(problem_ids_int)]

In [23]:
import pandas as pd
import pickle
import json
from bs4 import BeautifulSoup
from typing import Tuple, Dict, List, Set
import logging

In [24]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [25]:
def extract_question_kc_without_subset(
    assist_path: str,
    problem_bodies_path: str,
    keyid2idx_path: str = None,
    encoding: str = "ISO-8859-1",
    clean_html: bool = True
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """
    Extract question and KC data using only ASSIST dataset and Problem Bodies.
    No preprocessed subset needed!
    
    Args:
        assist_path: Path to ASSIST skill_builder_data_corrected_collapsed.csv
        problem_bodies_path: Path to ProblemBodies_23.csv
        keyid2idx_path: Optional path to keyid2idx.json for filtering specific KCs
        
    Returns:
        question_df: DataFrame with columns ['question_id', 'question_text']
        qid_to_kc: Dictionary mapping question_id to KC
    """
    
    # Step 1: Load ASSIST dataset
    logger.info(f"Loading ASSIST data from {assist_path}")
    assist_df = pd.read_csv(assist_path, encoding=encoding)
    
    # Step 2: Load Problem Bodies (full dataset)
    logger.info(f"Loading Problem Bodies from {problem_bodies_path}")
    pb_df = pd.read_csv(problem_bodies_path)
    
    # Step 3: If keyid2idx provided, filter to specific skills
    if keyid2idx_path:
        with open(keyid2idx_path, 'r') as f:
            keyid2idx = json.load(f)
        
        # Get the skills we care about
        target_skills = list(map(int, keyid2idx['concepts'].keys()))
        logger.info(f"Filtering to {len(target_skills)} skills from keyid2idx")
        
        # Filter ASSIST data to only these skills
        assist_df['skill_id'] = pd.to_numeric(assist_df['skill_id'], errors='coerce')
        assist_df = assist_df[assist_df['skill_id'].isin(target_skills)]
    
    # Step 4: Get unique problems from ASSIST that we need text for
    unique_problems = assist_df['problem_id'].unique()
    logger.info(f"Found {len(unique_problems)} unique problems in ASSIST data")
    
    # Step 5: Filter Problem Bodies to only problems in ASSIST
    pb_df['problem_id'] = pd.to_numeric(pb_df['problem_id'], errors='coerce')
    pb_subset = pb_df[pb_df['problem_id'].isin(unique_problems)]
    logger.info(f"Found {len(pb_subset)} problems with text in Problem Bodies")
    
    # Step 6: Create the question DataFrame
    question_data = []
    
    for _, row in pb_subset.iterrows():
        problem_id = int(row['problem_id'])
        
        # Get question text (assuming column is 'problem_body')
        if 'problem_body' in row:
            text = row['problem_body']
        elif 'body' in row:
            text = row['body']
        else:
            # Find the text column
            text_cols = [col for col in row.index if 'body' in col.lower() or 'text' in col.lower()]
            text = row[text_cols[0]] if text_cols else f"Problem {problem_id}"
        
        # Clean HTML if needed
        if clean_html and pd.notna(text):
            text = clean_html_text(text)
        
        question_data.append({
            'question_id': f'q{problem_id}',
            'question_text': text if pd.notna(text) else f"Problem {problem_id}"
        })
    
    question_df = pd.DataFrame(question_data)
    
    # Step 7: Create KC mapping
    problem_skill_map = assist_df[['problem_id', 'skill_id', 'skill_name']].drop_duplicates()
    
    qid_to_kc = {}
    for _, row in problem_skill_map.iterrows():
        if pd.notna(row['problem_id']) and pd.notna(row['skill_id']):
            qid = f"q{int(row['problem_id'])}"
            
            # Use skill name if available
            if pd.notna(row['skill_name']):
                kc_name = clean_kc_name(row['skill_name'])
            else:
                kc_name = f"kc_{int(row['skill_id'])}"
            
            # If problem already has a KC, you might want to handle multiple KCs
            if qid not in qid_to_kc:
                qid_to_kc[qid] = kc_name
    
    logger.info(f"\nExtraction complete:")
    logger.info(f"- Questions with text: {len(question_df)}")
    logger.info(f"- Question-KC mappings: {len(qid_to_kc)}")
    
    # Add any problems without text but with KC mappings
    questions_with_text = set(question_df['question_id'])
    questions_with_kc = set(qid_to_kc.keys())
    missing_text = questions_with_kc - questions_with_text
    
    if missing_text:
        logger.info(f"- Questions without text: {len(missing_text)}")
        # Add these with placeholder text
        missing_data = []
        for qid in missing_text:
            problem_id = int(qid[1:])  # Remove 'q' prefix
            missing_data.append({
                'question_id': qid,
                'question_text': f"Problem {problem_id} (no text available)"
            })
        
        question_df = pd.concat([question_df, pd.DataFrame(missing_data)], ignore_index=True)
    
    return question_df, qid_to_kc



In [26]:
def clean_html_text(text):
    """Remove HTML tags from text"""
    try:
        soup = BeautifulSoup(str(text), 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()
        cleaned = soup.get_text(separator=' ', strip=True)
        cleaned = ' '.join(cleaned.split())
        return cleaned
    except:
        return str(text)



In [27]:

def clean_kc_name(name):
    """Convert skill name to clean identifier"""
    clean = str(name).lower()
    clean = ''.join(c if c.isalnum() or c == ' ' else ' ' for c in clean)
    clean = '_'.join(clean.split())
    return clean.strip('_')



In [28]:
def extract_from_your_data_simplified():
    """Extract using only the essential files"""
    
    # Only need these two files
    assist_path = '/home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/raw_data/assist2009/skill_builder_data_corrected_collapsed.csv'
    pb_path = '/home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/data_subsets/ProblemBodies_23.csv'
    
    # Optional: if you want to filter to specific KCs
    keyid2idx_path = '/home/mahdi/Projects/pykt-toolkit-pt_emb/data/assist2009/keyid2idx.json'
    
    # Extract data
    question_df, qid_to_kc = extract_question_kc_without_subset(
        assist_path=assist_path,
        problem_bodies_path=pb_path,
        keyid2idx_path=keyid2idx_path  # Optional
    )
    
    print(f"\nExtracted {len(question_df)} questions")
    print(f"Sample questions:")
    print(question_df.head())
    
    print(f"\nSample KC mappings:")
    print(dict(list(qid_to_kc.items())[:5]))
    
    return question_df, qid_to_kc


In [29]:
# question_df, qid_to_kc = extract_from_your_data_simplified()


In [30]:
# question_df.head()


In [31]:
# qid_to_kc

In [32]:
# len(question_df),len(qid_to_kc)

In [33]:
# def save_question_data(question_df: pd.DataFrame, 
#                       qid_to_kc: Dict[str, str], 
#                       output_dir: str = "./data",
#                       format: str = "all") -> None:
#     """
#     Save question_df and qid_to_kc to files.
    
#     Args:
#         question_df: DataFrame with question_id and question_text
#         qid_to_kc: Dictionary mapping question_id to KC
#         output_dir: Directory to save files
#         format: "all", "csv", "pickle", or "json"
#     """
#     # Create output directory if it doesn't exist
#     os.makedirs(output_dir, exist_ok=True)
    
#     # Save question_df as CSV
#     if format in ["all", "csv"]:
#         csv_path = os.path.join(output_dir, "question_df.csv")
#         question_df.to_csv(csv_path, index=False)
#         print(f"Saved question_df to {csv_path}")
    
#     # Save qid_to_kc as pickle (most efficient for Python)
#     if format in ["all", "pickle"]:
#         pickle_path = os.path.join(output_dir, "qid_to_kc.pkl")
#         with open(pickle_path, 'wb') as f:
#             pickle.dump(qid_to_kc, f)
#         print(f"Saved qid_to_kc to {pickle_path}")
    
#     # Save qid_to_kc as JSON (human-readable)
#     if format in ["all", "json"]:
#         json_path = os.path.join(output_dir, "qid_to_kc.json")
#         with open(json_path, 'w') as f:
#             json.dump(qid_to_kc, f, indent=2)
#         print(f"Saved qid_to_kc to {json_path}")
    
#     # Alternative: Save both in a single pickle file
#     if format in ["all", "pickle"]:
#         combined_path = os.path.join(output_dir, "question_data_combined.pkl")
#         with open(combined_path, 'wb') as f:
#             pickle.dump({
#                 'question_df': question_df,
#                 'qid_to_kc': qid_to_kc
#             }, f)
#         print(f"Saved combined data to {combined_path}")


In [34]:
# save_question_data(question_df, qid_to_kc, output_dir="./my_data")


In [35]:
# def load_question_data(data_dir: str = "./data", 
#                       format: str = "pickle") -> Tuple[pd.DataFrame, Dict[str, str]]:
#     """
#     Load question_df and qid_to_kc from files.
    
#     Args:
#         data_dir: Directory containing the saved files
#         format: "pickle", "json", or "combined"
        
#     Returns:
#         question_df, qid_to_kc
#     """
#     # Load question_df from CSV
#     csv_path = os.path.join(data_dir, "question_df.csv")
#     if os.path.exists(csv_path):
#         question_df = pd.read_csv(csv_path)
#         print(f"Loaded question_df from {csv_path}")
#     else:
#         raise FileNotFoundError(f"Cannot find {csv_path}")
    
#     # Load qid_to_kc based on format
#     if format == "pickle":
#         pickle_path = os.path.join(data_dir, "qid_to_kc.pkl")
#         if os.path.exists(pickle_path):
#             with open(pickle_path, 'rb') as f:
#                 qid_to_kc = pickle.load(f)
#             print(f"Loaded qid_to_kc from {pickle_path}")
#         else:
#             raise FileNotFoundError(f"Cannot find {pickle_path}")
            
#     elif format == "json":
#         json_path = os.path.join(data_dir, "qid_to_kc.json")
#         if os.path.exists(json_path):
#             with open(json_path, 'r') as f:
#                 qid_to_kc = json.load(f)
#             print(f"Loaded qid_to_kc from {json_path}")
#         else:
#             raise FileNotFoundError(f"Cannot find {json_path}")
            
#     elif format == "combined":
#         combined_path = os.path.join(data_dir, "question_data_combined.pkl")
#         if os.path.exists(combined_path):
#             with open(combined_path, 'rb') as f:
#                 data = pickle.load(f)
#             question_df = data['question_df']
#             qid_to_kc = data['qid_to_kc']
#             print(f"Loaded combined data from {combined_path}")
#         else:
#             raise FileNotFoundError(f"Cannot find {combined_path}")
    
#     print(f"Loaded {len(question_df)} questions and {len(qid_to_kc)} KC mappings")
#     return question_df, qid_to_kc


In [36]:
# question_df2, qid_to_kc2 = load_question_data(data_dir="./my_data")


In [37]:
# question_df.equals(question_df2)


In [38]:
# qid_to_kc == qid_to_kc2

In [40]:
def create_mappings_from_question_csv(
    question_csv_path: str,
    assist_path: str,
    keyid2idx_path: str = None,
    encoding: str = "ISO-8859-1",
    save_dir: str = "./output"
) -> Tuple[Dict[str, str], Dict[str, int], pd.DataFrame]:
    """
    Create qid_to_kc and kc_name_to_id mappings based on a CSV file with question texts.
    
    Args:
        question_csv_path: Path to CSV with columns [question_id/problem_id, question_text]
        assist_path: Path to ASSIST dataset with skill mappings
        keyid2idx_path: Optional path to keyid2idx.json to ensure all KCs are included
        encoding: Encoding for reading files
        save_dir: Directory to save outputs
        
    Returns:
        qid_to_kc: Mapping from question_id to KC name
        kc_name_to_id: Mapping from KC name to skill_id
        question_df: DataFrame with questions and their KC mappings
    """
    
    # Load question CSV
    logger.info(f"Loading questions from {question_csv_path}")
    question_df = pd.read_csv(question_csv_path)
    
    # Standardize column names
    if 'problem_id' in question_df.columns:
        question_df = question_df.rename(columns={'problem_id': 'question_id'})
    
    # Ensure question_id is numeric for merging
    if question_df['question_id'].dtype == 'object' and question_df['question_id'].str.startswith('q').any():
        # Remove 'q' prefix if present
        question_df['numeric_id'] = question_df['question_id'].str.replace('q', '').astype(int)
    else:
        question_df['numeric_id'] = pd.to_numeric(question_df['question_id'], errors='coerce')
    
    # Load ASSIST dataset
    logger.info(f"Loading ASSIST data from {assist_path}")
    assist_df = pd.read_csv(assist_path, encoding=encoding)
    
    # Clean skill data
    assist_df['problem_id'] = pd.to_numeric(assist_df['problem_id'], errors='coerce')
    assist_df['skill_id'] = pd.to_numeric(assist_df['skill_id'], errors='coerce')
    
    # Get skill mappings for problems in question CSV
    valid_problem_ids = question_df['numeric_id'].dropna().unique()
    logger.info(f"Found {len(valid_problem_ids)} unique problems in question CSV")
    
    # Filter ASSIST to only these problems
    assist_filtered = assist_df[assist_df['problem_id'].isin(valid_problem_ids)]
    
    # Get unique problem-skill mappings (keeping first skill if multiple)
    problem_skill_map = assist_filtered.groupby('problem_id').first()[['skill_id', 'skill_name']].reset_index()
    
    # Create complete skill reference (all unique skills)
    all_skills = assist_df[['skill_id', 'skill_name']].drop_duplicates()
    all_skills = all_skills.dropna(subset=['skill_id'])
    
    # Initialize mappings
    kc_name_to_id = {}
    kc_id_to_name = {}
    
    # Process all skills to create kc_name_to_id
    skills_without_names = []
    
    for _, row in all_skills.iterrows():
        skill_id = int(row['skill_id'])
        skill_name = row['skill_name']
        
        # Only process skills that have names
        if pd.notna(skill_name):
            original_name = skill_name
            clean_name = clean_kc_name(original_name)
            
            # Handle duplicates
            if clean_name in kc_name_to_id and kc_name_to_id[clean_name] != skill_id:
                clean_name = f"{clean_name}_{skill_id}"
            
            kc_name_to_id[clean_name] = skill_id
            kc_id_to_name[skill_id] = clean_name
        else:
            # Track skills without names (we're skipping these)
            skills_without_names.append(skill_id)
    
    # Add any skills from keyid2idx if provided (only if they have names in ASSIST)
    if keyid2idx_path:
        with open(keyid2idx_path, 'r') as f:
            keyid2idx = json.load(f)
        
        for skill_id_str in keyid2idx.get('concepts', {}).keys():
            skill_id = int(skill_id_str)
            if skill_id not in kc_id_to_name and skill_id not in skills_without_names:
                # Check if this skill has a name in ASSIST
                skill_row = all_skills[all_skills['skill_id'] == skill_id]
                if not skill_row.empty and pd.notna(skill_row.iloc[0]['skill_name']):
                    original_name = skill_row.iloc[0]['skill_name']
                    clean_name = clean_kc_name(original_name)
                    kc_name_to_id[clean_name] = skill_id
                    kc_id_to_name[skill_id] = clean_name
    
    logger.info(f"Created mappings for {len(kc_name_to_id)} KCs with names")
    logger.info(f"Skipped {len(skills_without_names)} KCs without names")
    
    # Merge question data with skill mappings
    question_df = question_df.merge(
        problem_skill_map,
        left_on='numeric_id',
        right_on='problem_id',
        how='left'
    )
    
    # Create qid_to_kc mapping
    qid_to_kc = {}
    questions_with_kc = 0
    questions_without_kc = 0
    questions_with_unnamed_kc = 0
    
    for _, row in question_df.iterrows():
        # Get question ID (with 'q' prefix if not already)
        if 'question_id' in row and pd.notna(row['question_id']):
            qid = str(row['question_id'])
            if not qid.startswith('q'):
                qid = f"q{qid}"
        else:
            qid = f"q{int(row['numeric_id'])}"
        
        # Get KC for this question
        if pd.notna(row.get('skill_id')):
            skill_id = int(row['skill_id'])
            if skill_id in kc_id_to_name:
                kc_name = kc_id_to_name[skill_id]
                qid_to_kc[qid] = kc_name
                questions_with_kc += 1
            else:
                # Skill exists but has no name
                questions_with_unnamed_kc += 1
        else:
            questions_without_kc += 1
    
    logger.info(f"\nMapping results:")
    logger.info(f"- Questions with named KC: {questions_with_kc}")
    logger.info(f"- Questions with unnamed KC: {questions_with_unnamed_kc}")
    logger.info(f"- Questions without any KC: {questions_without_kc}")
    logger.info(f"- Total KCs with names: {len(kc_name_to_id)}")
    
    # Add KC name to dataframe
    question_df['kc_name'] = question_df['skill_id'].map(kc_id_to_name)
    
    # Save outputs
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # Save qid_to_kc
    with open(os.path.join(save_dir, 'qid_to_kc.pkl'), 'wb') as f:
        pickle.dump(qid_to_kc, f)
    
    with open(os.path.join(save_dir, 'qid_to_kc.json'), 'w') as f:
        json.dump(qid_to_kc, f, indent=2)
    
    # Save kc_name_to_id
    with open(os.path.join(save_dir, 'kc_name_to_id.json'), 'w') as f:
        json.dump(kc_name_to_id, f, indent=2)
    
    # Save complete mapping info
    mapping_info = {
        'kc_name_to_id': kc_name_to_id,
        'kc_id_to_name': kc_id_to_name,
        'total_questions': len(question_df),
        'questions_with_named_kc': questions_with_kc,
        'questions_with_unnamed_kc': questions_with_unnamed_kc,
        'questions_without_kc': questions_without_kc,
        'total_kcs_with_names': len(kc_name_to_id),
        'skills_without_names': skills_without_names
    }
    
    with open(os.path.join(save_dir, 'complete_mappings.json'), 'w') as f:
        json.dump(mapping_info, f, indent=2)
    
    # Save enhanced question dataframe
    question_df.to_csv(os.path.join(save_dir, 'questions_with_kc.csv'), index=False)
    
    logger.info(f"\nSaved all outputs to {save_dir}")
    
    return qid_to_kc, kc_name_to_id, question_df



In [41]:
def clean_kc_name(name: str) -> str:
    """Convert skill name to clean identifier format"""
    clean = str(name).lower()
    clean = ''.join(c if c.isalnum() or c == ' ' else ' ' for c in clean)
    clean = '_'.join(clean.split())
    while '__' in clean:
        clean = clean.replace('__', '_')
    return clean.strip('_')



In [42]:
def analyze_question_coverage(
    question_csv_path: str,
    assist_path: str,
    encoding: str = "ISO-8859-1"
) -> Dict:
    """
    Analyze how many questions have KC mappings and other statistics.
    """
    # Load data
    question_df = pd.read_csv(question_csv_path)
    assist_df = pd.read_csv(assist_path, encoding=encoding)
    
    # Get question IDs from CSV
    if 'problem_id' in question_df.columns:
        question_ids = pd.to_numeric(question_df['problem_id'], errors='coerce').dropna()
    else:
        question_ids = pd.to_numeric(question_df['question_id'].str.replace('q', ''), errors='coerce').dropna()
    
    # Get problem IDs from ASSIST
    assist_problems = set(assist_df['problem_id'].dropna().unique())
    
    # Calculate coverage
    questions_in_assist = set(question_ids) & assist_problems
    questions_not_in_assist = set(question_ids) - assist_problems
    
    # Get skill coverage
    problems_with_skills = set(
        assist_df[assist_df['skill_id'].notna()]['problem_id'].unique()
    )
    questions_with_skills = questions_in_assist & problems_with_skills
    
    # Check how many have skill names
    problems_with_skill_names = set(
        assist_df[assist_df['skill_id'].notna() & assist_df['skill_name'].notna()]['problem_id'].unique()
    )
    questions_with_skill_names = questions_in_assist & problems_with_skill_names
    
    stats = {
        'total_questions_in_csv': len(question_ids),
        'questions_in_assist': len(questions_in_assist),
        'questions_not_in_assist': len(questions_not_in_assist),
        'questions_with_skills': len(questions_with_skills),
        'questions_with_skill_names': len(questions_with_skill_names),
        'coverage_percentage': (len(questions_with_skill_names) / len(question_ids) * 100) if len(question_ids) > 0 else 0
    }
    
    return stats


In [43]:
question_csv = '/home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/data_subsets/assist2009/questions.csv'  # Your CSV with question_id and question_text
assist_data = '/home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/data_subsets/assist2009/skill_builder_data_corrected_collapsed.csv'    

In [44]:
stats = analyze_question_coverage(question_csv, assist_data)
print(f"\nCoverage Analysis:")
print(f"Total questions: {stats['total_questions_in_csv']}")
print(f"Questions with skills: {stats['questions_with_skills']}")
print(f"Questions with named skills: {stats['questions_with_skill_names']} ({stats['coverage_percentage']:.1f}%)")    


Coverage Analysis:
Total questions: 6141
Questions with skills: 6035
Questions with named skills: 5400 (87.9%)


/home/mahdi/miniconda3/envs/pykt/lib/python3.7/site-packages/ipykernel_launcher.py:1: DtypeWarning: Columns (17) have mixed types.Specify dtype option on import or set low_memory=False.
  """Entry point for launching an IPython kernel.


In [45]:
qid_to_kc, kc_name_to_id, question_df = create_mappings_from_question_csv(
        question_csv_path=question_csv,
        assist_path=assist_data,
        save_dir='./mappings_output'
    )

INFO:__main__:Loading questions from /home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/data_subsets/assist2009/questions.csv
INFO:__main__:Loading ASSIST data from /home/mahdi/Projects/pykt-toolkit-pt_emb/pretrained_Embedding/data_subsets/assist2009/skill_builder_data_corrected_collapsed.csv
/home/mahdi/miniconda3/envs/pykt/lib/python3.7/site-packages/ipykernel_launcher.py:4: DtypeWarning: Columns (17) have mixed types.Specify dtype option on import or set low_memory=False.
  after removing the cwd from sys.path.
INFO:__main__:Found 6035 unique problems in question CSV
INFO:__main__:Created mappings for 55 KCs with names
INFO:__main__:Skipped 5 KCs without names
INFO:__main__:
Mapping results:
INFO:__main__:- Questions with named KC: 4765
INFO:__main__:- Questions with unnamed KC: 635
INFO:__main__:- Questions without any KC: 741
INFO:__main__:- Total KCs with names: 55
INFO:__main__:
Saved all outputs to ./mappings_output


In [46]:
print(f"\nSample qid_to_kc mappings:")
for qid, kc in list(qid_to_kc.items())[:5]:
    skill_id = kc_name_to_id[kc]
    print(f"  {qid} -> {kc} (skill_id: {skill_id})")


Sample qid_to_kc mappings:
  q119299.0 -> probability_of_a_single_event (skill_id: 18)
  q97581.0 -> equation_solving_two_or_fewer_steps (skill_id: 311)
  q76339.0 -> area_trapezoid (skill_id: 297)
  q89068.0 -> interior_angles_figures_with_more_than_3_sides (skill_id: 21)
  q76309.0 -> area_trapezoid (skill_id: 297)
